# 2. The model

**Question:** how well can a simple model predict heart disease?

We use **logistic regression**: a simple model that gives each patient a risk between 0 and 1,
and whose decisions we can explain (notebook 4).

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score

data = pd.read_csv("../data/clean.csv")

features = ["age", "sex", "cp", "trestbps", "chol", "fbs",
            "restecg", "thalach", "exang", "oldpeak"]
X = data[features]
y = data["target"]

print("Patients:", len(X), " Sick:", y.sum())

## Prepare the features and build the model
- **Continuous** features (age, blood pressure, ...) are standardised: `(value − mean) / std`, so they are on the same scale.
- **Categorical** features (sex, chest pain type, ...) become 0/1 columns (one-hot), because their numbers are labels, not quantities.
- A **pipeline** puts the preparation and the model together, so they are always applied in the same way.

In [ ]:
continuous = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical = ["sex", "cp", "fbs", "restecg", "exang"]

prepare = ColumnTransformer([
    ("continuous", StandardScaler(), continuous),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
])

model = Pipeline([
    ("prepare", prepare),
    ("classifier", LogisticRegression(max_iter=1000)),
])

## Test the model fairly: cross-validation
We split the patients into 5 parts. Each time, the model learns on 4 parts and predicts the 5th.
At the end **every patient has a prediction from a model that never saw them**.
This is more reliable than a single train/test split, and it gives a prediction for all patients, which we need for the fairness check.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

risk = cross_val_predict(model, X, y, cv=folds, method="predict_proba")[:, 1]
predicted = (risk >= 0.5).astype(int)

print("Accuracy:", round(accuracy_score(y, predicted), 3))
print("AUC:     ", round(roc_auc_score(y, risk), 3))

- **Accuracy** = share of patients classified correctly (risk ≥ 0.5 → sick).
- **AUC** = probability that a random sick patient gets a higher risk than a random healthy one (0.5 = random guessing, 1 = perfect).

## Save the predictions for the fairness notebook

In [ ]:
data["risk"] = risk
data["predicted"] = predicted
data.to_csv("../data/predictions.csv", index=False)
print("Saved predictions")

## Train the final model on all patients and save it
This is the model used for the explanations and the app.

In [ ]:
model.fit(X, y)

bundle = {
    "model": model,
    "means": model[:-1].transform(X).mean(axis=0),            # average patient, after preparation
    "column_names": list(model[:-1].get_feature_names_out()),
}
joblib.dump(bundle, "../models/model.joblib")
print("Model saved")